# Sensitivity comparison: small ViT vs larger ViT

This notebook overlays GA-TETRIS sensitivity sweeps from two ViT models on the same axes:

- **Solid lines**: small ViT (`test_vit3.r160_in1k`)
- **Dashed lines**: larger ViT (`vit_wee_patch16_reg1_gap_256.sbb_in1k`)

Same color per algorithm, same axes, same x-tick values. Reader sees if the sensitivity profile is the same shape across models.

Configure the two `OUTPUT_FOLDER_*` paths below to point at your CSV folders.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

OUTPUT_FOLDER_SMALL  = "benchmark_csvs_vit_test_vit3.r160_in1k"
OUTPUT_FOLDER_LARGER = "benchmark_csvs_vit_vit_wee_patch16_reg1_gap_256.sbb_in1k"

LABEL_SMALL  = "Small ViT"
LABEL_LARGER = "Larger ViT"

ALG_COLORS = {
    "GA-TETRIS":             "#d62728",
    "Original TETRIS":       "#1f77b4",
    "Sort-by-Norm":          "#2ca02c",
    "Block-Wanda":           "#ff7f0e",
    "Random-Swaps":          "#8c564b",
    "Random-Swaps (sorted)": "#17becf",
}
def alg_color(a):
    return ALG_COLORS.get(a, "#555555")

ALG_DISPLAY = {
    "our_tetris":                       "GA-TETRIS",
    "original_tetris":                  "Original TETRIS",
    "block_wanda":                      "Block-Wanda",
    "sort_columns_by_norm":             "Sort-by-Norm",
    "random_swaps":                     "Random-Swaps",
    "random_swaps_find_mask":           "Random-Swaps",
    "random_swaps_sort_start":          "Random-Swaps (sorted)",
    "random_swaps_find_mask_sort_start":"Random-Swaps (sorted)",
}


In [ ]:
def load_one(folder, model_label):
    files = sorted(glob.glob(os.path.join(folder, "layer_metrics_*.csv")))
    if not files:
        print(f"WARNING: no CSVs found in {folder}")
        return pd.DataFrame()
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

    # Promote sort_start variant
    if "sort_start" in df.columns:
        is_rs = df["algorithm"].isin(["random_swaps", "random_swaps_find_mask"])
        df.loc[is_rs & (df["sort_start"] == True), "algorithm"] = (
            df.loc[is_rs & (df["sort_start"] == True), "algorithm"] + "_sort_start"
        )

    df["algorithm"] = df["algorithm"].map(ALG_DISPLAY).fillna(df["algorithm"])
    df["model"] = model_label
    return df


df_small  = load_one(OUTPUT_FOLDER_SMALL, LABEL_SMALL)
df_larger = load_one(OUTPUT_FOLDER_LARGER, LABEL_LARGER)
df = pd.concat([df_small, df_larger], ignore_index=True)

print(f"{LABEL_SMALL}:  {len(df_small)} rows from {df_small.algorithm.nunique() if len(df_small) else 0} algorithms")
print(f"{LABEL_LARGER}: {len(df_larger)} rows from {df_larger.algorithm.nunique() if len(df_larger) else 0} algorithms")

SPARSITY = 0.5
df_s = df[df["sparsity"] == SPARSITY].copy()


## Sensitivity comparison: MAX_ITER and RANDOM_SWAPS

Both panels share y-axis. Solid lines = small ViT. Dashed lines = larger ViT.

In [ ]:
SWEEP_BLOCK_W = 8

def get_sweeps(data, model_label):
    """Slice the data for a single model into the two GA-TETRIS sweep DataFrames
    plus the Original TETRIS data for MAX_ITER comparison."""
    sub = data[(data.model == model_label) & (data.block_cols == SWEEP_BLOCK_W)]
    ga = sub[sub.algorithm == "GA-TETRIS"]
    ot = sub[sub.algorithm == "Original TETRIS"]
    return {
        "ga_iter":  ga[ga.random_swaps == 100],
        "ga_swaps": ga[ga.max_iter == 10],
        "ot_iter":  ot,
    }

s_small  = get_sweeps(df_s, LABEL_SMALL)
s_larger = get_sweeps(df_s, LABEL_LARGER)

print("Small ViT:   GA iter sweep:", sorted(s_small['ga_iter'].max_iter.unique()),
      "  GA swaps sweep:", sorted(s_small['ga_swaps'].random_swaps.unique()),
      "  OT iter sweep:", sorted(s_small['ot_iter'].max_iter.unique()))
print("Larger ViT:  GA iter sweep:", sorted(s_larger['ga_iter'].max_iter.unique()),
      "  GA swaps sweep:", sorted(s_larger['ga_swaps'].random_swaps.unique()),
      "  OT iter sweep:", sorted(s_larger['ot_iter'].max_iter.unique()))


In [ ]:
def plot_mean(ax, data, x_col, label, color, marker="s", linestyle="-"):
    if len(data) == 0:
        return
    agg = (data.groupby(x_col)["score_improvement_pct"].mean()
                .reset_index().sort_values(x_col))
    ax.plot(agg[x_col], agg["score_improvement_pct"],
            marker=marker, linewidth=2.5, markersize=8,
            color=color, linestyle=linestyle, label=label)


fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

# ----------------------------------------------------------------------------
# LEFT PANEL: MAX_ITER (GA-TETRIS at RANDOM_SWAPS=100, plus Original TETRIS)
# ----------------------------------------------------------------------------
plot_mean(axes[0], s_small["ga_iter"], "max_iter",
          f"GA-TETRIS, {LABEL_SMALL}", alg_color("GA-TETRIS"),
          linestyle="-")
plot_mean(axes[0], s_larger["ga_iter"], "max_iter",
          f"GA-TETRIS, {LABEL_LARGER}", alg_color("GA-TETRIS"),
          linestyle="--")
plot_mean(axes[0], s_small["ot_iter"], "max_iter",
          f"Original TETRIS, {LABEL_SMALL}", alg_color("Original TETRIS"),
          linestyle="-")
plot_mean(axes[0], s_larger["ot_iter"], "max_iter",
          f"Original TETRIS, {LABEL_LARGER}", alg_color("Original TETRIS"),
          linestyle="--")

# x-ticks from data
all_iter_x = sorted(set(s_small["ga_iter"].max_iter.unique())
                    | set(s_larger["ga_iter"].max_iter.unique()))
axes[0].set_xticks(all_iter_x)
axes[0].xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x)}"))
axes[0].set_xlabel("MAX_ITER", fontsize=11)
axes[0].set_ylabel("Mean score improvement (%)", fontsize=11)
axes[0].set_title(f"MAX_ITER sensitivity  (block 1×{SWEEP_BLOCK_W})",
                  fontsize=12, fontweight="bold")
axes[0].grid(True, linestyle="--", alpha=0.35)
axes[0].legend(frameon=False, fontsize=9, loc="best")

# ----------------------------------------------------------------------------
# RIGHT PANEL: RANDOM_SWAPS (GA-TETRIS at MAX_ITER=10)
# ----------------------------------------------------------------------------
plot_mean(axes[1], s_small["ga_swaps"], "random_swaps",
          f"GA-TETRIS, {LABEL_SMALL}", alg_color("GA-TETRIS"),
          linestyle="-")
plot_mean(axes[1], s_larger["ga_swaps"], "random_swaps",
          f"GA-TETRIS, {LABEL_LARGER}", alg_color("GA-TETRIS"),
          linestyle="--")

axes[1].set_xscale("symlog", linthresh=1)
all_swap_x = sorted(set(s_small["ga_swaps"].random_swaps.unique())
                    | set(s_larger["ga_swaps"].random_swaps.unique()))
axes[1].set_xticks(all_swap_x)
axes[1].xaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))
axes[1].xaxis.set_minor_formatter(FuncFormatter(lambda x, _: ""))
axes[1].set_xlabel("RANDOM_SWAPS", fontsize=11)
axes[1].set_title(f"RANDOM_SWAPS sensitivity  (block 1×{SWEEP_BLOCK_W})",
                  fontsize=12, fontweight="bold")
axes[1].grid(True, linestyle="--", alpha=0.35)
axes[1].legend(frameon=False, fontsize=9, loc="best")

fig.tight_layout()
plt.show()


## Wall-clock comparison

For the cost numbers across both models — useful for thesis discussion of how the algorithms scale with model size.

In [ ]:
def cost_table(data, model_label):
    sub = data[(data.model == model_label) &
               (data.block_cols == SWEEP_BLOCK_W) &
               (data.algorithm == "GA-TETRIS")]
    return (sub.groupby(["max_iter", "random_swaps"])["total_time_sec"]
              .mean().round(3).reset_index()
              .rename(columns={"total_time_sec": f"time_{model_label}"}))

t_small  = cost_table(df_s, LABEL_SMALL)
t_larger = cost_table(df_s, LABEL_LARGER)

merged = t_small.merge(t_larger, on=["max_iter", "random_swaps"], how="outer")
print(f"\n=== GA-TETRIS wall-clock cost (s/layer) at block 1×{SWEEP_BLOCK_W} ===")
print(merged.to_string(index=False))
